In [1]:
pip install scipy seaborn

Note: you may need to restart the kernel to use updated packages.


In [1]:
"""
Exploratory data analysis and cohort characterisation for the OASIS-3
Alzheimer's disease prediction study.

Produces:
  - table1.csv / table1.txt : publication-ready demographic & clinical table
  - figures/*.png           : distribution and confound plots
  - console report          : statistical tests and confound warnings

Statistical tests:
  - continuous variables : Welch's t-test (unequal variance) + Mann-Whitney U
  - categorical variables: chi-square test of independence
"""
import os
import csv
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

COHORT = "oasis3_cohort.csv"
FIGDIR = "figures"
os.makedirs(FIGDIR, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ---------------------------------------------------------------- load
df = pd.read_csv(COHORT)

# numeric coercion
for c in ["age_at_scan", "age_at_entry", "education", "ses",
          "cdr", "mmse", "apoe_e4_count", "scan_day", "dx_gap_days"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["group"] = df["label"].map({0: "CN", 1: "AD"})
df["sex_label"] = df["sex"].map({1: "Male", 2: "Female"})
df["e4_carrier"] = (df["apoe_e4_count"] > 0).map({True: "Carrier", False: "Non-carrier"})

ad = df[df.label == 1]
cn = df[df.label == 0]

print("=" * 66)
print("COHORT OVERVIEW")
print("=" * 66)
print(f"  Total subjects        : {len(df)}")
print(f"  Alzheimer's disease   : {len(ad)} ({100*len(ad)/len(df):.1f}%)")
print(f"  Cognitively normal    : {len(cn)} ({100*len(cn)/len(df):.1f}%)")
print(f"  Class ratio (AD:CN)   : 1 : {len(cn)/len(ad):.2f}")

# ------------------------------------------------------- helper: tests
def cont_row(name, col, unit=""):
    """Continuous variable -> mean±sd per group, Welch t-test, Mann-Whitney."""
    a, c = ad[col].dropna(), cn[col].dropna()
    if len(a) < 2 or len(c) < 2:
        return None
    t, p_t = stats.ttest_ind(a, c, equal_var=False)
    u, p_u = stats.mannwhitneyu(a, c, alternative="two-sided")
    # Cohen's d (pooled sd)
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(c)-1)*c.std(ddof=1)**2)
                 / (len(a)+len(c)-2))
    d = (a.mean() - c.mean()) / sp if sp > 0 else np.nan
    return {
        "Variable": f"{name}{unit}",
        "AD (n=%d)" % len(ad): f"{a.mean():.1f} ± {a.std(ddof=1):.1f}",
        "CN (n=%d)" % len(cn): f"{c.mean():.1f} ± {c.std(ddof=1):.1f}",
        "Test": "Welch t / MWU",
        "p": f"{p_t:.2e}" if p_t < 0.001 else f"{p_t:.3f}",
        "p (MWU)": f"{p_u:.2e}" if p_u < 0.001 else f"{p_u:.3f}",
        "Effect size": f"d={d:.2f}",
        "_p_raw": p_t,
    }

def cat_row(name, col):
    """Categorical variable -> counts per group + chi-square."""
    tab = pd.crosstab(df[col], df["group"])
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return None
    chi2, p, dof, _ = stats.chi2_contingency(tab)
    ad_s = ", ".join(f"{ix}: {tab.loc[ix,'AD']} ({100*tab.loc[ix,'AD']/len(ad):.0f}%)"
                     for ix in tab.index)
    cn_s = ", ".join(f"{ix}: {tab.loc[ix,'CN']} ({100*tab.loc[ix,'CN']/len(cn):.0f}%)"
                     for ix in tab.index)
    return {
        "Variable": name,
        "AD (n=%d)" % len(ad): ad_s,
        "CN (n=%d)" % len(cn): cn_s,
        "Test": f"chi2(df={dof})",
        "p": f"{p:.2e}" if p < 0.001 else f"{p:.3f}",
        "p (MWU)": "",
        "Effect size": f"chi2={chi2:.1f}",
        "_p_raw": p,
    }

# ------------------------------------------------------------ Table 1
rows = []
for nm, col, unit in [("Age at scan", "age_at_scan", " (years)"),
                      ("Education", "education", " (years)"),
                      ("SES", "ses", ""),
                      ("MMSE", "mmse", ""),
                      ("CDR", "cdr", ""),
                      ("APOE e4 alleles", "apoe_e4_count", " (count)")]:
    r = cont_row(nm, col, unit)
    if r: rows.append(r)

for nm, col in [("Sex", "sex_label"),
                ("APOE e4 carrier", "e4_carrier"),
                ("Voxel size", "voxel"),
                ("Volume shape", "shape")]:
    r = cat_row(nm, col)
    if r: rows.append(r)

t1 = pd.DataFrame(rows)
t1_display = t1.drop(columns=["_p_raw"])
t1_display.to_csv("table1.csv", index=False)

print("\n" + "=" * 66)
print("TABLE 1 - COHORT CHARACTERISTICS")
print("=" * 66)
with open("table1.txt", "w") as fh:
    s = t1_display.to_string(index=False)
    print(s)
    fh.write(s + "\n")

# ------------------------------------------------------ confound check
print("\n" + "=" * 66)
print("CONFOUND ASSESSMENT")
print("=" * 66)
ALPHA = 0.05
warnings = []

for _, r in t1.iterrows():
    if r["_p_raw"] < ALPHA:
        var = r["Variable"]
        if "APOE" in var or "MMSE" in var or "CDR" in var:
            note = "EXPECTED - disease-related, not a confound"
        elif "Voxel" in var or "shape" in var.lower():
            note = "*** ACQUISITION CONFOUND - model may exploit protocol, not pathology"
        else:
            note = "POTENTIAL CONFOUND - consider matching or covariate adjustment"
        warnings.append((var, r["p"], note))

for var, p, note in warnings:
    print(f"  [{p:>9}]  {var:<22} {note}")
if not warnings:
    print("  No significant group differences detected.")

# explicit protocol-vs-diagnosis check
print("\n--- Acquisition protocol vs diagnosis ---")
prot = pd.crosstab(df["voxel"], df["group"])
prot["AD %"] = (100 * prot["AD"] / (prot["AD"] + prot["CN"])).round(1)
print(prot.to_string())
chi2, p_prot, dof, _ = stats.chi2_contingency(pd.crosstab(df["voxel"], df["group"]))
print(f"\n  chi2 = {chi2:.2f}, df = {dof}, p = {p_prot:.4f}")
if p_prot < ALPHA:
    print("  WARNING: acquisition protocol is associated with diagnosis.")
    print("           A model could learn scanner signature instead of disease.")
    print("           Mitigation: resample all volumes to a common voxel grid,")
    print("           and report protocol-stratified performance.")
else:
    print("  OK: no significant association between protocol and diagnosis.")

# ------------------------------------------------------ missing values
print("\n" + "=" * 66)
print("MISSING DATA")
print("=" * 66)
for c in ["age_at_scan", "education", "ses", "mmse", "cdr", "apoe_e4_count", "sex"]:
    if c in df.columns:
        n = df[c].isna().sum()
        if n:
            print(f"  {c:<18} {n:4d} missing ({100*n/len(df):.1f}%)")
print("  (variables not listed have no missing values)")

# ------------------------------------------------------------- figures
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

sns.histplot(data=df, x="age_at_scan", hue="group", kde=True, ax=axes[0,0])
axes[0,0].set_title("Age at scan"); axes[0,0].set_xlabel("Years")

sns.countplot(data=df, x="apoe_e4_count", hue="group", ax=axes[0,1])
axes[0,1].set_title("APOE e4 allele count"); axes[0,1].set_xlabel("e4 copies")

sns.countplot(data=df, x="sex_label", hue="group", ax=axes[0,2])
axes[0,2].set_title("Sex"); axes[0,2].set_xlabel("")

sns.boxplot(data=df, x="group", y="mmse", ax=axes[1,0])
axes[1,0].set_title("MMSE"); axes[1,0].set_xlabel("")

sns.boxplot(data=df, x="group", y="education", ax=axes[1,1])
axes[1,1].set_title("Education"); axes[1,1].set_xlabel("")

vc = pd.crosstab(df["voxel"], df["group"])
vc.plot(kind="bar", stacked=True, ax=axes[1,2])
axes[1,2].set_title("Acquisition protocol by group")
axes[1,2].set_xlabel(""); axes[1,2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(FIGDIR, "cohort_overview.png"), dpi=200)
plt.close()

# age vs diagnosis, split by APOE - checks whether age drives the APOE effect
plt.figure(figsize=(7, 5))
sns.violinplot(data=df, x="e4_carrier", y="age_at_scan", hue="group", split=True)
plt.title("Age distribution by APOE carrier status and diagnosis")
plt.tight_layout()
plt.savefig(os.path.join(FIGDIR, "age_apoe_interaction.png"), dpi=200)
plt.close()

print(f"\nFigures written to {FIGDIR}/")
print("Table written to table1.csv and table1.txt")

COHORT OVERVIEW
  Total subjects        : 365
  Alzheimer's disease   : 193 (52.9%)
  Cognitively normal    : 172 (47.1%)
  Class ratio (AD:CN)   : 1 : 0.89

TABLE 1 - COHORT CHARACTERISTICS
               Variable                                                                                                                                              AD (n=193)                                                                                                                                             CN (n=172)          Test        p  p (MWU) Effect size
    Age at scan (years)                                                                                                                                              76.8 ± 7.9                                                                                                                                             69.6 ± 8.6 Welch t / MWU 4.45e-15 5.48e-14      d=0.87
      Education (years)                                            

ValueError: List of boxplot statistics and `positions` values must have same the length